# Etapa 3 — Método analítico, acurácia e storytelling

Este notebook foi feito para a **Etapa 3** do projeto.  
Ele aplica um modelo simples de **Árvore de Decisão** a dados de prevalência de tabagismo e gera:

- treinamento e teste do modelo;
- medidas de acurácia;
- matriz de confusão;
- gráficos para o relatório;
- informações para o storytelling.

## Fonte dos dados
Neste notebook foi usado o link público indicado pela própria página da Our World in Data para o indicador **“Share of adults who smoke or use tobacco”**, que é **adaptado da World Health Organization – Global Health Observatory**.  
URL do CSV:
`https://ourworldindata.org/grapher/share-of-adults-who-smoke.csv?csvType=full&useColumnShortNames=false&v=1`

> Observação: a página da Our World in Data informa que esse indicador é baseado na **World Health Organization - Global Health Observatory (2024)** e cobre o período **2000–2022**.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

## 1. Carregar a base

In [ ]:
url = "https://ourworldindata.org/grapher/share-of-adults-who-smoke.csv?csvType=full&useColumnShortNames=false&v=1"
df = pd.read_csv(url)
df.head()

In [ ]:
df.columns.tolist()

## 2. Preparação dos dados

Nesta etapa:
- renomeamos colunas;
- removemos valores ausentes;
- criamos a variável-alvo em classes;
- transformamos variáveis categóricas em numéricas.


In [ ]:
# Ajuste simples dos nomes das colunas
df = df.rename(columns={
    "Entity": "pais",
    "Code": "codigo_pais",
    "Year": "ano",
    "Share of adults who smoke or use tobacco": "prevalencia"
})

# Mantém só as colunas principais
df = df[["pais", "codigo_pais", "ano", "prevalencia"]].copy()

# Remove linhas sem dados principais
df = df.dropna(subset=["pais", "ano", "prevalencia"])

# Remove agregados mundiais/regiões mais gerais, mantendo países com código de 3 letras
df = df[df["codigo_pais"].astype(str).str.len() == 3].copy()

df.head()

In [ ]:
# Criar classes para a prevalência
def classificar_prevalencia(x):
    if x <= 10:
        return "baixa"
    elif x <= 20:
        return "media"
    else:
        return "alta"

df["classe_prevalencia"] = df["prevalencia"].apply(classificar_prevalencia)

df["classe_prevalencia"].value_counts()

## 3. Codificação para o modelo

In [ ]:
le_pais = LabelEncoder()
df["pais_cod"] = le_pais.fit_transform(df["pais"])

# Variáveis de entrada
X = df[["pais_cod", "ano"]]

# Variável alvo
y = df["classe_prevalencia"]

X.head()

## 4. Divisão treino e teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

## 5. Treinamento do modelo

In [ ]:
modelo = DecisionTreeClassifier(max_depth=5, random_state=42)
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)

## 6. Medidas de acurácia

In [ ]:
acuracia = accuracy_score(y_test, y_pred)
print("Acurácia do modelo:", round(acuracia, 4))

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
matriz = confusion_matrix(y_test, y_pred)
matriz

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=matriz, display_labels=modelo.classes_)
disp.plot()
plt.title("Matriz de confusão - Árvore de Decisão")
plt.show()

## 7. Gráficos para o relatório

In [ ]:
# Distribuição das classes
df["classe_prevalencia"].value_counts().plot(kind="bar")
plt.title("Distribuição das classes de prevalência")
plt.xlabel("Classe")
plt.ylabel("Quantidade")
plt.show()

In [ ]:
# Média da prevalência por ano
media_ano = df.groupby("ano")["prevalencia"].mean()

plt.figure(figsize=(8,5))
plt.plot(media_ano.index, media_ano.values, marker="o")
plt.title("Média da prevalência de tabagismo por ano")
plt.xlabel("Ano")
plt.ylabel("Prevalência média")
plt.show()

In [ ]:
# Visualização simples da árvore
plt.figure(figsize=(16,8))
plot_tree(modelo, feature_names=X.columns, class_names=modelo.classes_, filled=True, fontsize=8)
plt.title("Árvore de Decisão")
plt.show()

## 8. Resultados preliminares para usar no relatório

In [ ]:
print("Resumo para o relatório:")
print("- Total de registros usados:", len(df))
print("- Quantidade de países:", df['pais'].nunique())
print("- Período analisado:", int(df['ano'].min()), "a", int(df['ano'].max()))
print("- Acurácia:", round(acuracia, 4))

## 9. Texto-base para a Etapa 3

Depois de rodar o notebook, você pode atualizar o relatório com frases como:

- O modelo de Árvore de Decisão foi aplicado à base de dados escolhida.
- A base utilizada possui dados de `X` países no período de `ano inicial` a `ano final`.
- A acurácia obtida pelo modelo foi de `valor`.
- A matriz de confusão mostrou `interpretação curta`.
- O modelo se mostrou adequado para fins preliminares e poderá ser melhorado com novas variáveis.

## Observação importante
Este modelo é **simples e acadêmico**, adequado para trabalho de faculdade.  
Ele usa `pais` e `ano` como variáveis de entrada para prever a classe de prevalência.  
Em um projeto mais avançado, seria ideal incluir também mais indicadores de política pública do tabaco.
